# Target Transform × Loss 직접 검증 — LGBM 단일 회귀

**목적**: 트리 모델(LGBM)에서 `손실함수 × y 변환` 16조합을 동일 HP·동일 PP·동일 KFold로 비교하여, 신규 노트북 통일 정책 결정의 실측 근거 확보.

**검증할 가설**:
1. **트리 + 분포 손실(poisson/tweedie)**: 변환 OFF (none)이 best — 분포 가정 충돌 회피
2. **트리 + RMSE(regression)**: 변환 무관 — 거의 동일 (y 작아서 효과 미미)
3. **분포 손실 + log1p/yeo/quantile**: 충돌로 악화

**16 조합 매트릭스**:

| loss \ transform | none | log1p | yeo-johnson | quantile |
|---|---|---|---|---|
| regression | • | • | • | • |
| poisson | • | • | • | • |
| tweedie_1.2 | • | • | • | • |
| tweedie_1.5 | • | • | • | • |

**구조**:
- 학습: die-level broadcast (Y unit → 4 die, 단순 회귀)
- KFold 5, unit-level
- 후처리 없이 die→unit mean 집계만
- HP는 1차 lgbm best (poisson 컨텍스트, 다른 loss에도 그대로 적용)
- 16 조합 × 5 fold = 80 모델 → ~2시간 예상

**입력**: `0_data/compet_xs_data.csv`, `compet_ys_*_data.csv`
**출력**: 화면 출력만 (영구 산출물 없음)

**스코프**: 트리(LGBM)만. **ElasticNet은 별도 — `02_reg_single`/`03_two_stage`에 실험축으로 추가**.

## 1. 환경 설정 + import

In [7]:
import os, sys
RESUME = True   # 기존 Optuna study(db)에 이어서 학습할지 (필요시 config 셀에서 덮어씀)
# ── Colab이면 코드 번들 1개(code.zip)만 받아 풀기 — 데이터·경로·폰트는 setup.py가 처리 ──
try:
    import google.colab  # Colab에서만 import 성공
    GDRIVE_CODE_ID = '1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I'  # code.zip = setup.py+requirements+utils+2_preprocessing+3_modeling 지원코드
    if not os.path.exists('/content/project/setup.py'):
        os.system('pip -q install gdown')
        os.system(f'gdown {GDRIVE_CODE_ID} -O /content/code.zip')
        os.system('unzip -qo /content/code.zip -d /content/project')
    os.chdir('/content/project')
except ImportError:
    pass
# ── 공통: cwd에서 위로 setup.py(+utils/)를 자동탐색해 실행 (노트북 깊이·드라이브 위치 무관) ──
_d = os.getcwd()
while not (os.path.exists(os.path.join(_d, 'setup.py')) and os.path.isdir(os.path.join(_d, 'utils'))):
    _p = os.path.dirname(_d)
    if _p == _d:
        raise RuntimeError('프로젝트 루트(setup.py + utils/)를 못 찾음 — cwd 확인')
    _d = _p
if _d not in sys.path:
    sys.path.insert(0, _d)
import runpy
runpy.run_path(os.path.join(_d, 'setup.py'))

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from utils.config import PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, OUTPUT_DIR
from utils.data import load_all, get_feat_cols, split_xs

PP_DIR = os.path.join(PROJECT_ROOT, '2_preprocessing')
if PP_DIR not in sys.path:
    sys.path.insert(0, PP_DIR)
MOD_DIR = os.path.join(PROJECT_ROOT, '3_modeling')
if MOD_DIR not in sys.path:
    sys.path.insert(0, MOD_DIR)

from modules import preprocess

import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.preprocessing import PowerTransformer, QuantileTransformer

import logging, time
logging.getLogger('lightgbm').setLevel(logging.ERROR)

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'lightgbm v{lgb.__version__}')


setup 완료
PROJECT_ROOT = c:\Users\Dell5371\Desktop\기업연계프로젝트
lightgbm v4.6.0


## 2. 설정 (PP_FIXED + LGBM HP + 매트릭스 축)

- PP_FIXED: strategy_common §1 트리 공통
- LGBM HP: 1차 lgbm best (poisson 컨텍스트). loss/transform만 흔들고 나머진 동일 고정
- N_JOBS: 14 (단일 노트북 전용)
- 매트릭스: LOSSES (4) × TRANSFORMS (4) = 16

In [8]:
N_FOLDS = 5
N_JOBS  = 14
CLIP_Y_EXTREME = True

PP_FIXED = {
    'missing_threshold':          0.30,
    'corr_threshold':             0.90,
    'corr_keep_by':               'std',
    'add_indicator':              True,
    'indicator_threshold':        0.05,
    'spatial_max_dist':           6.0,
    'post_impute_corr_threshold': 0.96,
    'post_impute_corr_keep_by':   'std',
}

# 1차 lgbm best (poisson 컨텍스트, OOF=0.005521)
LGBM_HP = {
    'n_estimators':      957,
    'learning_rate':     0.00602,
    'num_leaves':        379,
    'max_depth':         10,
    'min_child_samples': 343,
    'subsample':         0.711,
    'subsample_freq':    1,
    'colsample_bytree':  0.597,
    'reg_alpha':         0.00840,
    'reg_lambda':        0.000151,
    'min_split_gain':    1.016e-04,
    'path_smooth':       24.81,
    'random_state':      SEED,
    'n_jobs':            N_JOBS,
    'verbose':           -1,
}

# 매트릭스 축
LOSSES     = ['regression', 'poisson', 'tweedie_1.2', 'tweedie_1.5']
TRANSFORMS = ['none', 'log1p', 'yeo', 'quantile']

print(f'N_FOLDS={N_FOLDS} | N_JOBS={N_JOBS}')
print(f'LGBM HP: 1차 lgbm best ({len(LGBM_HP)} keys)')
print(f'매트릭스: {len(LOSSES)} losses × {len(TRANSFORMS)} transforms = {len(LOSSES)*len(TRANSFORMS)} 조합')
print(f'  losses    : {LOSSES}')
print(f'  transforms: {TRANSFORMS}')


N_FOLDS=5 | N_JOBS=14
LGBM HP: 1차 lgbm best (15 keys)
매트릭스: 4 losses × 4 transforms = 16 조합
  losses    : ['regression', 'poisson', 'tweedie_1.2', 'tweedie_1.5']
  transforms: ['none', 'log1p', 'yeo', 'quantile']


## 3. 데이터 로드 + PP_FIXED 1회 적용

In [9]:
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)

ys_input = {k: v.copy() for k, v in ys.items()}
if CLIP_Y_EXTREME:
    y_raw = ys_input['train'][TARGET_COL]
    second_max = y_raw[y_raw < y_raw.max()].max()
    n_clipped = (y_raw >= 1.0).sum()
    ys_input['train'][TARGET_COL] = y_raw.clip(upper=second_max)
    print(f'[CLIP_Y_EXTREME] 1.0 → {second_max:.6f} clip, {n_clipped}개')

pp = preprocess.run(xs, ys_input, feat_cols, xs_dict, params=PP_FIXED)
xs_train = pp['xs_train']
feat_cols_clean = pp['feat_cols']

X_train_die = xs_train[feat_cols_clean].values.astype(np.float64)
uid_train_die = xs_train[KEY_COL].values

y_train_unit = ys_input['train'].set_index(KEY_COL)[TARGET_COL]
y_train_die_broadcast = pd.Series(uid_train_die).map(y_train_unit).values.astype(np.float64)
assert not pd.isna(y_train_die_broadcast).any()

n_train_die = len(X_train_die)
print(f'\n[PP] feat_cols_clean={len(feat_cols_clean)}, X_train_die={X_train_die.shape}')
print(f'  unit train: {len(y_train_unit):,}')
print(f'  y range: [{y_train_die_broadcast.min():.6f}, {y_train_die_broadcast.max():.6f}]')
print(f'  y mean : {y_train_die_broadcast.mean():.6f}, zero ratio: {(y_train_die_broadcast==0).mean():.3f}')


Xs: (174572, 1091)  |  Ys: train=26,187, val=8,727, test=8,729
[CLIP_Y_EXTREME] 1.0 → 0.097417 clip, 1개
[Stage 0] 웨이퍼맵 사전 제외: 1087 → 1033 (54개 제거)
클리닝 파이프라인 시작
원본 feature 수: 1033
[상수/극저분산 제거] threshold=1e-06
  제거: 105개, 잔여: 928개
    컬럼: 1033 → 928 (105개 제거)
    DataFrame: (104748, 986)

[고결측 제거] threshold=30%
  제거: 5개, 잔여: 923개
    컬럼: 928 → 923 (5개 제거)
    DataFrame: (104748, 981)

[중복 컬럼 제거] sample_n=5000
  제거: 27개, 잔여: 896개
    컬럼: 923 → 896 (27개 제거)
    DataFrame: (104748, 954)

[고상관 제거] threshold=0.9, keep_by=std (std)
  제거: 332개, 잔여: 564개
    컬럼: 896 → 564 (332개 제거)
    DataFrame: (104748, 622)

[결측 indicator] 9개 컬럼 추가 (결측률 >= 5%)
[공간 보간 imputation] 총 결측: 343,494
  train-only 모드: train 104,748 / 전체 174,572 행
  1단계 (공간 보간, dist<=6.0): 161,870개 채움 → 잔여: 181,624
  2단계 (lot 평균, train 기준): 100,428개 채움 → 잔여: 81,196
  3단계 (train 전체 평균): 81,196개 채움 → 잔여: 0

  [요약] 343,494 → 공간(161,870) → lot(100,428) → 전체(81,196) → 잔여(0)

[고상관 제거] threshold=0.96, keep_by=std (std)
  제거: 0개, 잔여: 564개
    

## 4. K-fold split (unit-level)

In [10]:
unique_units = y_train_unit.index.values
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(kf.split(unique_units))
print(f'fold split: {N_FOLDS} folds, unit 단위, seed={SEED}')


fold split: 5 folds, unit 단위, seed=42


## 5. 학습·평가 helper

- `_build_transformer(name, y_fit)` — 변환 함수 (forward, inverse) 튜플 반환. **fold별 train fold y에 fit** (leakage 방지)
- `train_one(loss, transform_name)` — 5-fold OOF unit RMSE 반환

**변환 종류**:
| name | 처리 |
|---|---|
| `none` | identity (변환 없음) |
| `log1p` | `np.log1p` / `np.expm1` (고정) |
| `yeo` | `PowerTransformer(method='yeo-johnson')` — `lambda` 자동 추정 |
| `quantile` | `QuantileTransformer(output_distribution='normal')` — rank 기반 |

**fold별 fit 이유**: train fold의 y로만 transformer를 학습해야 OOF가 valid (val fold의 y가 transformer에 새어들면 leakage).

In [11]:
def _build_params(loss):
    p = dict(LGBM_HP)
    if loss.startswith('tweedie'):
        p['objective'] = 'tweedie'
        p['tweedie_variance_power'] = float(loss.split('_')[1])
    else:
        p['objective'] = loss
    return p


def _build_transformer(transform_name, y_fit_arr):
    """fold별 train fold y에 fit한 (forward_fn, inverse_fn) 반환."""
    if transform_name == 'none':
        return (lambda y: y, lambda y: y)
    if transform_name == 'log1p':
        return (np.log1p, np.expm1)
    if transform_name == 'yeo':
        pt = PowerTransformer(method='yeo-johnson', standardize=False)
        pt.fit(y_fit_arr.reshape(-1, 1))
        return (
            lambda y: pt.transform(np.asarray(y).reshape(-1, 1)).ravel(),
            lambda y: pt.inverse_transform(np.asarray(y).reshape(-1, 1)).ravel(),
        )
    if transform_name == 'quantile':
        n_q = min(1000, len(y_fit_arr))
        qt = QuantileTransformer(output_distribution='normal', n_quantiles=n_q, random_state=SEED)
        qt.fit(y_fit_arr.reshape(-1, 1))
        return (
            lambda y: qt.transform(np.asarray(y).reshape(-1, 1)).ravel(),
            lambda y: qt.inverse_transform(np.asarray(y).reshape(-1, 1)).ravel(),
        )
    raise ValueError(f'unknown transform: {transform_name}')


def _mean_die_to_unit(pred_die, uid_die):
    df = pd.DataFrame({KEY_COL: uid_die, 'pred': pred_die})
    return df.groupby(KEY_COL, sort=False)['pred'].mean()


def train_one(loss, transform_name):
    """5-fold OOF unit RMSE 반환. 음수 호환 안 되면 (None, elapsed) 리턴."""
    params = _build_params(loss)
    is_dist_loss = (loss == 'poisson' or loss.startswith('tweedie'))
    oof_die_pred = np.full(n_train_die, np.nan)

    t0 = time.time()
    for fold_idx, (tr_uidx, vl_uidx) in enumerate(FOLDS):
        tr_units = unique_units[tr_uidx]
        vl_units = unique_units[vl_uidx]
        tr_mask = np.isin(uid_train_die, tr_units)
        vl_mask = np.isin(uid_train_die, vl_units)

        y_tr_raw = y_train_die_broadcast[tr_mask]
        forward_fn, inverse_fn = _build_transformer(transform_name, y_tr_raw)
        y_fit = forward_fn(y_tr_raw)

        # 분포 손실(poisson/tweedie)은 음수 target 불허 → INCOMPAT
        if is_dist_loss and (y_fit < 0).any():
            return None, time.time() - t0

        model = lgb.LGBMRegressor(**params)
        model.fit(X_train_die[tr_mask], y_fit)

        pred_raw = model.predict(X_train_die[vl_mask])
        pred = inverse_fn(pred_raw)
        pred = np.clip(pred, 0.0, None)
        oof_die_pred[vl_mask] = pred

    assert not np.isnan(oof_die_pred).any()
    oof_unit = _mean_die_to_unit(oof_die_pred, uid_train_die)
    oof_unit = oof_unit.reindex(y_train_unit.index)
    rmse = float(np.sqrt(np.mean((oof_unit.values - y_train_unit.values) ** 2)))
    elapsed = time.time() - t0
    return rmse, elapsed

print('helpers 정의 완료 (분포 손실+음수 변환은 INCOMPAT 처리)')


helpers 정의 완료 (분포 손실+음수 변환은 INCOMPAT 처리)


## 6. 16 조합 실행 (loss × transform)

진행상황 실시간 출력. 16 × 5 = 80 모델 학습 → ~2시간 예상.

In [12]:
# 기존 results 보존 (재실행 시 끝난 조합 skip)
try:
    results
except NameError:
    results = {}

total = len(LOSSES) * len(TRANSFORMS)
done = 0

print(f'{"loss":15s} {"transform":>10s}  {"oof_rmse":>10s}  {"elapsed":>8s}  {"prog":>8s}')
print('-' * 70)
overall_t0 = time.time()
for loss in LOSSES:
    for transform_name in TRANSFORMS:
        done += 1
        prog = f'{done}/{total}'
        if (loss, transform_name) in results:
            existing = results[(loss, transform_name)]
            tag = f'{existing:10.6f}' if existing is not None else f'{"INCOMPAT":>10s}'
            print(f'{loss:15s} {transform_name:>10s}  {tag}  {"":>8s}  {prog:>8s}  (skip)')
            continue
        rmse, elapsed = train_one(loss, transform_name)
        results[(loss, transform_name)] = rmse
        tag = f'{rmse:10.6f}' if rmse is not None else f'{"INCOMPAT":>10s}'
        print(f'{loss:15s} {transform_name:>10s}  {tag}  {elapsed:>6.0f}s  {prog:>8s}')
print('-' * 70)
print(f'전체 완료 ({time.time()-overall_t0:.0f}s)')


loss             transform    oof_rmse   elapsed      prog
----------------------------------------------------------------------
regression            none    0.005523                1/16  (skip)
regression           log1p    0.005523                2/16  (skip)
regression             yeo    0.005814                3/16  (skip)
regression        quantile    0.006123                4/16  (skip)
poisson               none    0.005521                5/16  (skip)
poisson              log1p    0.005521                6/16  (skip)
poisson                yeo    0.005785                7/16  (skip)
poisson           quantile    INCOMPAT      49s      8/16
tweedie_1.2           none    0.005522     413s      9/16
tweedie_1.2          log1p    0.005522     400s     10/16
tweedie_1.2            yeo    0.005788     386s     11/16
tweedie_1.2       quantile    INCOMPAT      47s     12/16
tweedie_1.5           none    0.005527     365s     13/16
tweedie_1.5          log1p    0.005527     437s     1

## 7. 결과 요약 + 가설 검증

In [13]:
# 4 × 4 매트릭스 표
print('=' * 90)
print('  loss × transform 매트릭스 — OOF unit RMSE')
print('=' * 90)
header = f'{"loss":15s}  ' + '  '.join(f'{t:>10s}' for t in TRANSFORMS) + f'  {"best":>10s}'
print(header)
print('-' * 90)
for loss in LOSSES:
    row_vals = [results.get((loss, t)) for t in TRANSFORMS]
    valid = [(t, v) for t, v in zip(TRANSFORMS, row_vals) if v is not None]
    best_t = min(valid, key=lambda tv: tv[1])[0] if valid else 'N/A'
    cells = []
    for v in row_vals:
        cells.append(f'{v:10.6f}' if v is not None else f'{"INCOMPAT":>10s}')
    row_str = f'{loss:15s}  ' + '  '.join(cells) + f'  {best_t:>10s}'
    print(row_str)
print('=' * 90)

# valid 결과만 가지고 베스트 + 평균 계산
valid_results = {k: v for k, v in results.items() if v is not None}
if valid_results:
    best = min(valid_results.items(), key=lambda kv: kv[1])
    (best_loss, best_t), best_rmse = best
    print(f'\n[BEST 조합] loss={best_loss}, transform={best_t}, OOF RMSE={best_rmse:.6f}')

# transform별 평균 (valid 결과만)
print(f'\n[transform별 평균] (valid 결과만)')
for t in TRANSFORMS:
    valid_vs = [results[(l, t)] for l in LOSSES if results.get((l, t)) is not None]
    if valid_vs:
        avg = float(np.mean(valid_vs))
        print(f'  {t:>10s}: {avg:.6f}  (n={len(valid_vs)})')
    else:
        print(f'  {t:>10s}: ALL INCOMPAT')

# 가설 H1: 분포 손실 + 변환 비교
print(f'\n[가설 H1] 분포 손실(poisson + tweedie) + 변환 비교')
dist_losses = ['poisson', 'tweedie_1.2', 'tweedie_1.5']
dist_avgs = {}
for t in TRANSFORMS:
    valid_vs = [results[(l, t)] for l in dist_losses if results.get((l, t)) is not None]
    incompat_n = len(dist_losses) - len(valid_vs)
    if valid_vs:
        avg = float(np.mean(valid_vs))
        dist_avgs[t] = avg
        print(f'  {t:>10s}: {avg:.6f}  (n={len(valid_vs)}, INCOMPAT={incompat_n})')
    else:
        print(f'  {t:>10s}: ALL INCOMPAT')
if dist_avgs:
    dist_best_t = min(dist_avgs, key=dist_avgs.get)
    print(f'  → 분포 손실 best transform: {dist_best_t}')
    print(f"  → 'none' 또는 'log1p'이면 H1 지지 (충돌 가설 확정)")

# 가설 H2: regression + 변환 무관?
print(f'\n[가설 H2] regression(분포 무관) + 변환 비교')
reg_vals = {t: results.get(('regression', t)) for t in TRANSFORMS}
for t in TRANSFORMS:
    v = reg_vals[t]
    print(f'  {t:>10s}: {v:.6f}' if v is not None else f'  {t:>10s}: N/A')
valid_reg = [v for v in reg_vals.values() if v is not None]
if len(valid_reg) >= 2:
    reg_range = max(valid_reg) - min(valid_reg)
    print(f'  → range: {reg_range:.6f}')
    print(f"  → range가 0.0001 미만이면 H2 지지 (변환 효과 미미)")

# 정책 결정 가이드
print(f'\n[정책 결정 가이드]')
if dist_avgs:
    print(f'  - 분포 손실(poisson/tweedie) best transform = {min(dist_avgs, key=dist_avgs.get)}')
reg_valid = {t: v for t, v in reg_vals.items() if v is not None}
if reg_valid:
    print(f'  - regression best transform              = {min(reg_valid, key=reg_valid.get)}')
if valid_results:
    print(f'  - 전체 best 조합                          = {best_loss} + {best_t}')
print(f'  → 이 결과로 strategy_common.md 트리계열 정책 확정')


  loss × transform 매트릭스 — OOF unit RMSE
loss                   none       log1p         yeo    quantile        best
------------------------------------------------------------------------------------------
regression         0.005523    0.005523    0.005814    0.006123       log1p
poisson            0.005521    0.005521    0.005785    INCOMPAT        none
tweedie_1.2        0.005522    0.005522    0.005788    INCOMPAT       log1p
tweedie_1.5        0.005527    0.005527    0.005801    INCOMPAT        none

[BEST 조합] loss=poisson, transform=none, OOF RMSE=0.005521

[transform별 평균] (valid 결과만)
        none: 0.005523  (n=4)
       log1p: 0.005523  (n=4)
         yeo: 0.005797  (n=4)
    quantile: 0.006123  (n=1)

[가설 H1] 분포 손실(poisson + tweedie) + 변환 비교
        none: 0.005524  (n=3, INCOMPAT=0)
       log1p: 0.005524  (n=3, INCOMPAT=0)
         yeo: 0.005791  (n=3, INCOMPAT=0)
    quantile: ALL INCOMPAT
  → 분포 손실 best transform: none
  → 'none' 또는 'log1p'이면 H1 지지 (충돌 가설 확정)

[가설 H2] regre